In [ ]:
# Imports
import random
import os
import json
import re
from pathlib import Path
import tqdm
import cv2
from scipy import ndimage


import pandas as pd
import numpy as np

from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns

from scripts.preprocess import *
from scripts.pt_model import *
from scripts.layoutlmv3_model import *
from scripts.basic_model import *

from scripts.donut_model import *
from scripts.donut_training_utils import *

In [ ]:
ground_truth_df = pd.read_csv("../finalproject_data/cleaned_invoices.csv")
combined_results = pd.read_csv("../finalproject_data/combined_results.csv")

print(ground_truth_df.columns)
print(combined_results.columns)

In [ ]:
full_df = ground_truth_df.merge(
    combined_results[["original_file", "original_path"]],
    left_on="File Name",
    right_on="original_file",
    how="inner"
)

# # Sample from full dataset to speed up training
# sample_df = full_df.iloc[:100, ]

# # Step 1: train / holdout
# train_df, holdout_df = train_test_split(sample_df, test_size=0.2, random_state=42)

# # Step 2: train / val
# train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

# # Step 3: test = holdout
# test_df = holdout_df

# Tiny overfit subset: 4 to 8 examples
tiny_df = full_df.sample(n=6, random_state=42).reset_index(drop=True)

# For an overfit test, keep train/val/test extremely small
train_df = tiny_df.copy()
val_df = tiny_df.copy()
test_df = tiny_df.copy()

display(tiny_df[["File Name", "original_path", "invoice_number", "invoice_date", "seller_name", "client_name", "tax", "net_worth", "total_amount"]])

print(train_df.columns)
print(train_df[["original_path", "invoice_number"]].head())
print("Rows:", len(train_df))

In [ ]:
# TRAIN DONUT MODEL
train_output = train_donut_invoice_model(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir="../finalproject_data/donut_finetuned_model",
    image_col="original_path",
    augment_factor=1
)

print("Training complete.")
print("Validation metrics:", train_output["validation_metrics"])
print("Test metrics:", train_output["test_metrics"])


In [ ]:
logs = train_output["trainer"].state.log_history

train_losses = [x["loss"] for x in logs if "loss" in x]
eval_losses = [x["eval_loss"] for x in logs if "eval_loss" in x]

print("\n--- Training Loss (last 5) ---")
print(train_losses[-5:])

if eval_losses:
    print("\n--- Eval Loss (last 5) ---")
    print(eval_losses[-5:])

In [ ]:
# LOAD TRAINED MODEL
donut_detector = DonutInvoiceTextDetector(
    output_dir="../finalproject_data/donut_output"
)

donut_detector.reload_model("../finalproject_data/donut_finetuned_model")

In [ ]:
result = donut_detector.process_single_image(train_df.iloc[0]["original_path"])
print(result["raw_sequence"])

In [ ]:
# RUN INFERENCE
summary_df = donut_detector.run_inference(
    test_df,
    image_path_col="original_path", 
    sample_frac=1,
    batch_size=4
)

print(summary_df.head())

In [ ]:
donut_metrics_df, donut_overall = donut_detector.evaluate_against_ground_truth(test_df)

print(donut_metrics_df)
print("\nOverall Metrics:", donut_overall)

In [ ]:
dashboard_stats = create_analysis_dashboard(
    donut_detector.full_results,
    metrics_df=donut_metrics_df,
    fields=[
        "invoice_number",
        "invoice_date",
        "seller_name",
        "client_name",
        "tax",
        "net_worth",
        "total_amount"
    ]
)

In [ ]:
visualize_sample_results(
    donut_detector.full_results,
    visualize_text_fn=None,   # Donut doesn’t use bounding boxes
    n_samples=3
)